In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader

# 1. Setup Device (M2 Chip Support)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Online Augmentation Pipeline
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)), # Standard size for pre-trained models
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # Handles lighting variance
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # Standard ImageNet values
])

Using device: mps


In [8]:
# 3. Load Pre-trained Model
# We use weights from ImageNet (millions of general images)
model = models.resnet18(weights='IMAGENET1K_V1')

# 4. "Freeze" the Backbone
# This prevents the model from forgetting basic shapes/textures
for param in model.parameters():
    param.requires_grad = False

# 5. Replace the "Head"
# We change the last layer to output 2 classes (e.g., Tripos vs. Other)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2) 

model = model.to(device)

# 6. Loss and Optimizer
criterion = nn.CrossEntropyLoss()
# Only optimize the parameters of the final layer (the head)
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

In [9]:
import os
from torchvision.datasets import ImageFolder

# Define the path to your folders
data_dir = './data' 

# Load the datasets
train_dataset = ImageFolder(root=os.path.join(data_dir, 'train'), transform=data_transforms)
val_dataset = ImageFolder(root=os.path.join(data_dir, 'val'), transform=data_transforms)

# Create DataLoaders (this feeds the images into your M2 GPU in batches)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

print(f"Classes found: {train_dataset.classes}") # Should output ['other', 'tripos']
print(f"Folder Mapping: {train_dataset.class_to_idx}")

Classes found: ['other', 'tripos']
Folder Mapping: {'other': 0, 'tripos': 1}


In [ ]:
import time
import copy

def train_phytoplankton_model(model, criterion, optimizer, num_epochs=10):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # Set model to training mode
                dataloader = train_loader
            else:
                model.eval()   # Set model to evaluate mode
                dataloader = val_loader

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data
            for inputs, labels in dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Zero the parameter gradients
                optimizer.zero_grad()

                # Forward pass
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward pass + optimize
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # Statistics
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.float() / len(dataloader.dataset)

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Deep copy the model if it's the most accurate one so far
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')

    # Load best model weights
    model.load_state_dict(best_model_wts)
    return model



In [11]:
# To start training:
model = train_phytoplankton_model(model, criterion, optimizer, num_epochs=11)

Epoch 0/10
----------
train Loss: 0.8263 Acc: 0.5741
val Loss: 0.6819 Acc: 0.5526

Epoch 1/10
----------
train Loss: 0.8051 Acc: 0.4444
val Loss: 0.8132 Acc: 0.4474

Epoch 2/10
----------
train Loss: 0.5497 Acc: 0.6852
val Loss: 0.5508 Acc: 0.7105

Epoch 3/10
----------
train Loss: 0.5889 Acc: 0.6296
val Loss: 0.5197 Acc: 0.7368

Epoch 4/10
----------
train Loss: 0.4571 Acc: 0.8333
val Loss: 0.6018 Acc: 0.6053

Epoch 5/10
----------
train Loss: 0.4232 Acc: 0.8704
val Loss: 0.5226 Acc: 0.6842

Epoch 6/10
----------
train Loss: 0.3673 Acc: 0.8519
val Loss: 0.4393 Acc: 0.8158

Epoch 7/10
----------
train Loss: 0.3250 Acc: 0.8704
val Loss: 0.4815 Acc: 0.6842

Epoch 8/10
----------
train Loss: 0.3146 Acc: 0.8889
val Loss: 0.4131 Acc: 0.7895

Epoch 9/10
----------
train Loss: 0.2578 Acc: 0.9630
val Loss: 0.4111 Acc: 0.8158

Epoch 10/10
----------
train Loss: 0.2289 Acc: 0.9444
val Loss: 0.3633 Acc: 0.8684

Training complete in 0m 8s
Best val Acc: 0.868421


In [12]:
torch.save(model.state_dict(), 'tripos_classifier_v4.pth')